# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 06: ColPali: OCR-Free Multimodal Document Retrieval from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In enterprise document processing (PDF financial statements, engineering manuals, legal contracts):
- Traditional RAG uses a fragile multi-stage pipeline: PDF $\to$ Optical Character Recognition (OCR) $\to$ Text Chunking $\to$ Dense Embedding.
- **Catastrophic Failure Modes:** OCR completely mangles complex financial tables, loses font hierarchies (headers vs footnotes), and ignores charts, graphs, and logos.

In 2024, Faysse et al. introduced **ColPali (ColBERT + PaliGemma)**:
- **Zero OCR Pipeline!**
- Directly inputs the **raw image of the PDF page** into a Vision-Language Model!
- Generates multivector token patch embeddings.
- Retrieves documents using **ColBERT Late Interaction MaxSim** between text query tokens and visual page patch embeddings!

---

### 2. Mathematical Foundations: ColPali MaxSim
Given text query tokens $Q = \{q_1, \dots, q_L\}$ and visual page patch tokens $P = \{p_1, \dots, p_M\}$:

$$\text{Score}(Q, \text{PageImage}) = \sum_{i=1}^L \max_{j=1}^M \left( q_i \cdot p_j^T \right)$$

If the user queries: *"What was Q3 Cloud Revenue?"*:
- Query token `Revenue` directly aligns with the visual image patch containing the revenue row in the balance sheet table, bypassing text parsing errors entirely!

In [ ]:
import numpy as np

np.random.seed(42)
dim = 4  # Patch embedding dimension

# Step 1: Simulate Query Tokens ("quarterly", "revenue", "table")
query_tokens = ["quarterly", "revenue", "table"]
E_query = np.random.randn(len(query_tokens), dim)
E_query /= np.linalg.norm(E_query, axis=1, keepdims=True)

# Step 2: Simulate 2 PDF Page Images as Visual Patch Embeddings
# Page 1: A financial table containing revenue charts (4 patches)
E_page1 = np.random.randn(4, dim)
# Make patch 1 strongly align with "revenue" and patch 2 with "table"
E_page1[1] = E_query[1] + 0.05 * np.random.randn(dim) # Revenue patch
E_page1[2] = E_query[2] + 0.05 * np.random.randn(dim) # Table patch
E_page1 /= np.linalg.norm(E_page1, axis=1, keepdims=True)

# Page 2: General legal terms & conditions page without tables (4 patches)
E_page2 = np.random.randn(4, dim)
E_page2 /= np.linalg.norm(E_page2, axis=1, keepdims=True)

# Step 3: Implement ColPali Multi-Vector MaxSim Scoring
def colpali_maxsim(query_vecs, page_patch_vecs):
    # Cross-similarity matrix: (num_query_tokens, num_page_patches)
    sim_matrix = query_vecs @ page_patch_vecs.T
    # Find maximum patch match for each query word
    max_matches = np.max(sim_matrix, axis=1)
    # Sum over all query words
    return np.sum(max_matches), sim_matrix

score_page1, sim_mat1 = colpali_maxsim(E_query, E_page1)
score_page2, sim_mat2 = colpali_maxsim(E_query, E_page2)

print("--- ColPali Multimodal Retrieval Results ---")
print(f"Page 1 (Visual Financial Table) Score: {score_page1:.4f}")
print(f"Page 2 (Plain Legal Text Page) Score:  {score_page2:.4f}")
print(f"Relevance Margin: {(score_page1 - score_page2):.4f} (Page 1 ranked #1!)")

print("\nPage 1 Token-to-Patch Match Matrix:")
for q_idx, word in enumerate(query_tokens):
    best_patch = np.argmax(sim_mat1[q_idx])
    print(f"  Query Token '{word:<10}' aligned with Visual Patch #{best_patch} (Score: {np.max(sim_mat1[q_idx]):.3f})")

### 3. Step-by-Step Code Walkthrough
1. **Vision-Language Mapping:** The query tokens were directly compared against visual image patch vectors.
2. **Fine-Grained Alignment:** The word "revenue" precisely located the table cell patch on Page 1 ($0.999$ similarity).
3. **OCR-Free Advantage:** Charts, logos, and complex tabular borders are natively indexed without lossy text extraction.

---

### 4. Industry Applications & Trade-offs
- **Enterprise PDF RAG:** High-accuracy search over scanned insurance claims, financial prospectuses, and engineering blueprints.
- **Trade-off:** High vector storage footprint: storing 500 patch vectors per PDF page requires multi-vector indexing databases (Qdrant / Vespa).

---

### 5. Top Interview Questions & Answers
**Q1: Why is ColPali considered a paradigm shift over traditional RAG?**
> **Answer:** Traditional RAG uses a brittle multi-component pipeline: PDF parsers, OCR, layout detectors, chunking, and text embedding models. ColPali unifies the entire pipeline into an end-to-end vision-language model: it takes raw page screenshots, encodes visual and textual information jointly, and retrieves pages via Late Interaction MaxSim without any OCR preprocessing.

**Q2: What vision-language backbone does ColPali use?**
> **Answer:** ColPali is fine-tuned from **PaliGemma-3B** (SigLIP vision encoder + Gemma language model). It maps the $14 \times 14$ visual patch output tokens of SigLIP through PaliGemma's projection layers to generate 128-dimensional multi-vector embeddings per page.